In [1]:
import pandas as pd

In [2]:
MDSs  = pd.read_csv(
    '/archive/bioinformatics/Zhou_lab/shared/jjin/project/steins_gate/meqtl/dataset/original/Olafur_2024/Data-S3.tab',
    sep='\t',
    header=0,
    dtype=str
)

In [3]:
maf_thresh = 0.05  # 最低次等位基因频率
fdr_thresh = 0.05  # 最大 FDR
hap_thresh = 20    # 最少信息型单倍型数
pval_thresh = 1e-5
numeric_cols = [
    'SeqVariant_MAF',
    'n_haplotypes',
    'SeqVariant_mCpG_effectsize',
    'SeqVariant_5mCpG_pvalue',
    'SeqVariant_start',
    'SeqVariant_end',
    'MDS_start',
    'MDS_end',
    'MDS_ref_methylrate',
    'MDS_alt_methylrate',
]
for col in numeric_cols:
    MDSs[col] = pd.to_numeric(MDSs[col], errors='coerce')

ci = (
    MDSs['SeqVariant_MDS_95CI']
    .str.strip('[]')
    .str.split(',', expand=True)
    .astype(float)
)
MDSs['CI_lower'] = ci[0]
MDSs['CI_upper'] = ci[1]

# 只保留SNP SeqVariant_alt的长度只为1

MDSs_filtered = MDSs[
    (MDSs['SeqVariant_MAF'] >= maf_thresh) &
    # (df['FDR'] < fdr_thresh) &
    # 置信区间完全在正或负一侧
    ((MDSs['CI_lower'] > 0) | (MDSs['CI_upper'] < 0)) &
    (MDSs['n_haplotypes'] >= hap_thresh) &
    (MDSs['SeqVariant_rank'] == 'primary') &
    (MDSs['SeqVariant_5mCpG_pvalue'] < pval_thresh) &
    ((MDSs['SeqVariant_start'] - MDSs['MDS_start'])< 9000) &
    ((MDSs['SeqVariant_start'] - MDSs['MDS_end'])>-9000) &
    (((MDSs['MDS_ref_methylrate'] - MDSs['MDS_alt_methylrate'])>0.5) |
    ((MDSs['MDS_ref_methylrate'] - MDSs['MDS_alt_methylrate'])<-0.5)) &
    (MDSs['SeqVariant_alt'].str.len() == 1) &
    (MDSs['SeqVariant_ref'].str.len() == 1)
]

print(f"原始条目：{len(MDSs)}，筛选后保留：{len(MDSs_filtered)}")

原始条目：86252，筛选后保留：524


In [4]:
# 计算effect size 用差值
# Use .loc to avoid SettingWithCopyWarning
MDSs_filtered = MDSs_filtered.copy()
MDSs_filtered['SeqVariant_effect_size'] = (
    MDSs_filtered['MDS_alt_methylrate']
  - MDSs_filtered['MDS_ref_methylrate']
).round(4)


In [5]:
MDSs_filtered.head()

,Chrom,SeqVariant_start,SeqVariant_end,SeqVariant_name,SeqVariant_rsname,SeqVariant_ref,SeqVariant_alt,SeqVariant_minor_allele,SeqVariant_MAF,SeqVariant_LD_count,...,SeqVariant_MDS_95CI,SeqVariant_5mCpG_pvalue,n_haplotypes,GWAS_disease,GWAS_disease_markers,GWAS_other.traits,GWAS_other.traits_markers,CI_lower,CI_upper,SeqVariant_effect_size
205,chr1,2969824,2969824,chr1:2969824:SG,rs78232041,T,C,ALT,0.1620,1,...,"1.69,1.76",0.0,12984,NaN,NaN,NaN,NaN,1.69,1.76,0.530
288,chr1,3902180,3902180,chr1:3902180:SG,rs10797355,T,C,ALT,0.3120,24,...,"1.51,1.55",0.0,12917,NaN,NaN,NaN,NaN,1.51,1.55,0.510
558,chr1,7536406,7536406,chr1:7536406:SG,rs72865422,C,T,ALT,0.0569,11,...,"1.98,2.09",0.0,13082,NaN,NaN,NaN,NaN,1.98,2.09,0.732
639,chr1,8898949,8898949,chr1:8898949:SG,rs112653519,C,T,ALT,0.2270,17,...,"1.5,1.56",0.0,12579,NaN,NaN,NaN,NaN,1.50,1.56,0.530
1378,chr1,20851974,20851974,chr1:20851974:SG,rs61781078,C,T,ALT,0.3700,447,...,"1.43,1.48",0.0,11481,NaN,NaN,"C-reactive-protein-measurement,chronotype-meas...","chr1:21108040:SG(0.013|0.81),chr1:20874832:SG(...",1.43,1.48,0.540


In [74]:
# 只保存 MDSs_filtered 的Chrom, SeqVariant_start, SeqVariant_end, SeqVariant_ref, SeqVariant_alt, MDS_start, MDS_end, SeqVariant_effect_size
# change MDS_start name to CPG_region_start, MDS_end name to CPG_region_end, SeqVariant_start to SNP_region_start, SeqVariant_end to SNP_region_end, SeqVariant_ref to SNP_ref, SeqVariant_alt to SNP_alt, SeqVariant_effect_size to effect_size, Chrom to chrom
MDSs_filtered_save  = MDSs_filtered.copy()
MDSs_filtered_save.rename(columns={'MDS_start': 'CPG_region_start', 'MDS_end': 'CPG_region_end', 'SeqVariant_start': 'SNP_region_start', 'SeqVariant_end': 'SNP_region_end', 'SeqVariant_ref': 'SNP_ref', 'SeqVariant_alt': 'SNP_alt', 'SeqVariant_effect_size': 'effect_size', 'Chrom': 'chrom'}, inplace=True)

MDSs_filtered_save = MDSs_filtered_save[['chrom', 'SNP_region_start', 'SNP_region_end', 'SNP_ref', 'SNP_alt', 'CPG_region_start', 'CPG_region_end', 'effect_size']]
MDSs_filtered_save['CPG_region_start'] = MDSs_filtered_save['CPG_region_start'] - 1
MDSs_filtered_save['CPG_region_end'] = MDSs_filtered_save['CPG_region_end'] + 1
MDSs_filtered_save.head()


,chrom,SNP_region_start,SNP_region_end,SNP_ref,SNP_alt,CPG_region_start,CPG_region_end,effect_size
205,chr1,2969824,2969824,T,C,2969761,2970042,0.530
288,chr1,3902180,3902180,T,C,3903667,3903823,0.510
558,chr1,7536406,7536406,C,T,7536166,7536444,0.732
639,chr1,8898949,8898949,C,T,8898877,8898975,0.530
1378,chr1,20851974,20851974,C,T,20851873,20852222,0.540


In [75]:
MDSs_filtered_save.shape

(524, 8)

In [77]:
import sys

sys.path.append("/archive/bioinformatics/Zhou_lab/shared/jjin/project/steins/test_code")
from selene_mini import *
genome = Genome(input_path='/archive/bioinformatics/Zhou_lab/shared/jzhou/graphseq/Homo_sapiens.GRCh38.dna.primary_assembly.fa')

In [82]:
def encoding2seq(encoding):
    seq = ""
    # ACGT
    for i in range(encoding.shape[0]):
        if encoding[i][0] == 1:
            seq += "A"
        elif encoding[i][1] == 1:
            seq += "C"
        elif encoding[i][2] == 1:
            seq += "G"
        elif encoding[i][3] == 1:
            seq += "T"
        else:
            seq += "N"  # Unknown nucleotide
    return seq


# calculate cpg number between CPG_region_start and CPG_region_end
for index, row in MDSs_filtered_save.iterrows():
    seq_encoding = genome.get(row['chrom'], int(row['CPG_region_start']), int(row['CPG_region_end']))
    # seq is [1000, 4] encoding
    seq = encoding2seq(seq_encoding)
    # count cpg number
    cpg_number = seq.count("CG")
    # print(cpg_number)
    MDSs_filtered_save.loc[index, 'cpg_number'] = cpg_number

In [ ]:
ii = 0
for index, row in MDSs_filtered_save.iterrows():
    seq_encoding = genome.get(row['chrom'], int(row['CPG_region_start']), int(row['CPG_region_end']))
    # seq is [1000, 4] encoding
    seq = encoding2seq(seq_encoding)
    # count cpg number
    cpg_number = seq.count("CG")
    # print(cpg_number)
    MDSs_filtered_save.loc[index, 'cpg_number'] = cpg_number
    # print(seq, cpg_number)
    # ii += 1
    # if ii > 100:
    #     break

In [80]:
MDSs_filtered_save

,chrom,SNP_region_start,SNP_region_end,SNP_ref,SNP_alt,CPG_region_start,CPG_region_end,effect_size,cpg_number
205,chr1,2969824,2969824,T,C,2969761,2970042,0.530,13.0
288,chr1,3902180,3902180,T,C,3903667,3903823,0.510,4.0
558,chr1,7536406,7536406,C,T,7536166,7536444,0.732,7.0
639,chr1,8898949,8898949,C,T,8898877,8898975,0.530,8.0
1378,chr1,20851974,20851974,C,T,20851873,20852222,0.540,9.0
...,...,...,...,...,...,...,...,...,...
85419,chr9,122219390,122219390,C,T,122226019,122226506,-0.770,NaN
85429,chr9,122406599,122406599,C,T,122406108,122406841,-0.791,NaN
85662,chr9,128030899,128030899,A,G,128030790,128030860,0.570,NaN
85752,chr9,129762680,129762680,G,A,129762610,129762846,0.550,NaN


In [83]:
import os
if not os.path.exists('/archive/bioinformatics/Zhou_lab/shared/jjin/project/steins_gate/meqtl/dataset/processed/Olafur_2024_fix'):
    os.makedirs('/archive/bioinformatics/Zhou_lab/shared/jjin/project/steins_gate/meqtl/dataset/processed/Olafur_2024_fix')
# if os.path.exists('/archive/bioinformatics/Zhou_lab/shared/jjin/project/steins_gate/meqtl/dataset/processed/Olafur_2024/MDSs.csv'):
#     os.remove('/archive/bioinformatics/Zhou_lab/shared/jjin/project/steins_gate/meqtl/dataset/processed/Olafur_2024/MDSs.csv')
MDSs_filtered_save.to_csv('/archive/bioinformatics/Zhou_lab/shared/jjin/project/steins_gate/meqtl/dataset/processed/Olafur_2024_fix/MDSs.csv', index=False)

In [50]:
CPG_units  = pd.read_csv(
    '/archive/bioinformatics/Zhou_lab/shared/jjin/project/steins_gate/meqtl/dataset/original/Olafur_2024/Data-S1.tab',
    sep='\t',
    header=0,
    dtype=str
)

In [51]:
maf_thresh = 0.05  # 最低次等位基因频率
fdr_thresh = 0.05  # 最大 FDR
hap_thresh = 20    # 最少信息型单倍型数
pval_thresh = 1e-5
numeric_cols = [
    'SeqVariant_MAF',
    'n_haplotypes',
    'SeqVariant_mCpG_effectsize',
    'SeqVariant_5mCpG_pvalue',
    'SeqVariant_start',
    'SeqVariant_end',
    'CpG_start',
    'CpG_end',
    'CpG_ref_methylrate',
    'CpG_alt_methylrate',
]
for col in numeric_cols:
    CPG_units[col] = pd.to_numeric(CPG_units[col], errors='coerce')

ci = (
    CPG_units['SeqVariant_5mCpG_95CI']
    .str.strip('[]')
    .str.split(',', expand=True)
    .astype(float)
)
CPG_units['CI_lower'] = ci[0]
CPG_units['CI_upper'] = ci[1]

CPG_units_filtered = CPG_units[
    (CPG_units['SeqVariant_MAF'] >= maf_thresh) &
    # (df['FDR'] < fdr_thresh) &
    # 置信区间完全在正或负一侧
    ((CPG_units['CI_lower'] > 0) | (CPG_units['CI_upper'] < 0)) &
    (CPG_units['n_haplotypes'] >= hap_thresh) &
    (CPG_units['SeqVariant_rank'] == 'primary') &
    (CPG_units['SeqVariant_5mCpG_pvalue'] < pval_thresh) &
    ((CPG_units['SeqVariant_start'] - CPG_units['CpG_start'])< 9000) &
    ((CPG_units['SeqVariant_start'] - CPG_units['CpG_end'])>-9000) &
    (((CPG_units['CpG_ref_methylrate'] - CPG_units['CpG_alt_methylrate'])>0.5) |
    ((CPG_units['CpG_ref_methylrate'] - CPG_units['CpG_alt_methylrate'])<-0.5)) &
    (CPG_units['SeqVariant_alt'].str.len() == 1) &
    (CPG_units['SeqVariant_ref'].str.len() == 1) &
    (CPG_units['SeqVariant_alt'].str != "*")
]

print(f"原始条目：{len(CPG_units)}，筛选后保留：{len(CPG_units_filtered)}")

原始条目：1669152，筛选后保留：8715


In [52]:
# 计算effect size 用差值
# Use .loc to avoid SettingWithCopyWarning
CPG_units_filtered = CPG_units_filtered.copy()
CPG_units_filtered['SeqVariant_effect_size'] = (
    CPG_units_filtered['CpG_alt_methylrate']
  - CPG_units_filtered['CpG_ref_methylrate']
).round(4)

In [72]:
CPG_units_filtered_save  = CPG_units_filtered.copy()
CPG_units_filtered_save.rename(columns={'CpG_start': 'CPG_region_start', 'CpG_end': 'CPG_region_end', 'SeqVariant_start': 'SNP_region_start', 'SeqVariant_end': 'SNP_region_end', 'SeqVariant_ref': 'SNP_ref', 'SeqVariant_alt': 'SNP_alt', 'SeqVariant_effect_size': 'effect_size', 'Chrom': 'chrom'}, inplace=True)

CPG_units_filtered_save = CPG_units_filtered_save[['chrom', 'SNP_region_start', 'SNP_region_end', 'SNP_ref', 'SNP_alt', 'CPG_region_start', 'CPG_region_end', 'effect_size']]
CPG_units_filtered_save = CPG_units_filtered_save[CPG_units_filtered_save["SNP_alt"] != '*']
CPG_units_filtered_save['CPG_region_start'] = CPG_units_filtered_save['CPG_region_start'] - 1
CPG_units_filtered_save['CPG_region_end'] = CPG_units_filtered_save['CPG_region_start'] + 2
CPG_units_filtered_save.head()

,chrom,SNP_region_start,SNP_region_end,SNP_ref,SNP_alt,CPG_region_start,CPG_region_end,effect_size
45,chr1,901812,901812,A,G,904371.0,904373.0,0.530
59,chr1,904947,904947,G,A,904314.0,904316.0,-0.510
855,chr1,1173815,1173815,A,G,1174140.0,1174142.0,0.520
1737,chr1,1420203,1420203,C,A,1420011.0,1420013.0,0.550
1738,chr1,1420203,1420203,C,A,1420030.0,1420032.0,0.514


In [86]:
ii = 0
for index, row in CPG_units_filtered_save.iterrows():
    seq_encoding = genome.get(row['chrom'], int(row['CPG_region_start']), int(row['CPG_region_end']))
    # seq is [1000, 4] encoding
    seq = encoding2seq(seq_encoding)
    # count cpg number
    cpg_number = seq.count("CG")
    # print(cpg_number)
    CPG_units_filtered_save.loc[index, 'cpg_number'] = cpg_number

    # print(seq, cpg_number)
    # ii += 1
    # if ii > 100:
    #     break
CPG_units_filtered_save = CPG_units_filtered_save[CPG_units_filtered_save['cpg_number'] == 1]

In [87]:
if not os.path.exists('/archive/bioinformatics/Zhou_lab/shared/jjin/project/steins_gate/meqtl/dataset/processed/Olafur_2024_fix'):
    os.makedirs('/archive/bioinformatics/Zhou_lab/shared/jjin/project/steins_gate/meqtl/dataset/processed/Olafur_2024_fix')
# if os.path.exists('/archive/bioinformatics/Zhou_lab/shared/jjin/project/steins_gate/meqtl/dataset/processed/Olafur_2024/MDSs.csv'):
#     os.remove('/archive/bioinformatics/Zhou_lab/shared/jjin/project/steins_gate/meqtl/dataset/processed/Olafur_2024/MDSs.csv')
CPG_units_filtered_save.to_csv('/archive/bioinformatics/Zhou_lab/shared/jjin/project/steins_gate/meqtl/dataset/processed/Olafur_2024_fix/CPG_units.csv', index=False)